In [1]:
import polars as pl
import re
import spacy
import warnings
import pandas as pd
import numpy as np
from requests.packages.urllib3.exceptions import InsecureRequestWarning
from unidecode import unidecode
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Attention, Concatenate
from tensorflow.keras.models import Model, load_model



C:\Users\gaye\Anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [ ]:
class DataPreparation:
    """
    A class used to clean, tokenize, and prepare English and French sentence data for machine learning models.
    
    Attributes:
    ----------
    file_path : str
        The file path of the CSV file containing the data.
    en_model : str
        The SpaCy model to use for English tokenization.
    fr_model : str
        The SpaCy model to use for French tokenization.
    df : DataFrame
        The dataframe containing the loaded data.
    nlp_en : SpaCy Model
        The loaded SpaCy model for English.
    nlp_fr : SpaCy Model
        The loaded SpaCy model for French.
    """

    def __init__(self, file_path, en_model='en_core_web_sm', fr_model='fr_core_news_sm'):
        """
        Initializes the DataPreparation class by loading the data and SpaCy models.
        
        Parameters:
        ----------
        file_path : str
            The path to the CSV file containing the data.
        en_model : str, optional
            The SpaCy model to use for English tokenization (default is 'en_core_web_sm').
        fr_model : str, optional
            The SpaCy model to use for French tokenization (default is 'fr_core_news_sm').
        """
        self.file_path = file_path
        self.df = pl.read_csv(file_path, n_rows=100000)  #Load a subset of the data for processing

        warnings.simplefilter('ignore', InsecureRequestWarning)
        
        self.nlp_en = spacy.load(en_model)
        self.nlp_fr = spacy.load(fr_model) 

    def clean_text(self, text):
        """
        Cleans the input text by converting it to lowercase, removing diacritics, and removing special characters.
        
        Parameters:
        ----------
        text : str
            The text to be cleaned.
        
        Returns:
        ----------
        str
            The cleaned text.
        """
        if text is not None:
            text = text.lower()                #Convert to lowercase
            text = unidecode(text)             #Remove diacritics, using unicode convention
            text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  #Remove special characters
        return text

    def tokenize_text(self, text, nlp):
        """
        Tokenizes the input text using the specified SpaCy model.
        
        Parameters:
        ----------
        text : str
            The text to be tokenized.
        nlp : SpaCy Model
            The SpaCy model to use for tokenization.
        
        Returns:
        ----------
        str
            The tokenized text.
        """
        if text is not None:
            doc = nlp(text)
            return ' '.join([token.text for token in doc])
        return text

    def add_start_end_tokens(self, text):
        """
        Adds <start> and <end> tokens to the input text.
        
        Parameters:
        ----------
        text : str
            The text to which <start> and <end> tokens will be added.
        
        Returns:
        ----------
        str
            The text with <start> and <end> tokens added.
        """
        if text is not None:
            return f"<start> {text} <end>"
        return text

    def prepare_data(self):
        """
        Prepares the data by cleaning, tokenizing, and adding <start> and <end> tokens to the English and French sentences.
        """
        #Clean the 'English words/sentences' and 'French words/sentences' columns
        self.df = self.df.with_columns([
            pl.col("English words/sentences").apply(self.clean_text).alias("en_cleaned"),
            pl.col("French words/sentences").apply(self.clean_text).alias("fr_cleaned")
        ])
        
        #Tokenize the cleaned columns
        self.df = self.df.with_columns([
            pl.col("en_cleaned").apply(lambda text: self.tokenize_text(text, self.nlp_en)).alias("en_tokenized"),
            pl.col("fr_cleaned").apply(lambda text: self.tokenize_text(text, self.nlp_fr)).alias("fr_tokenized")
        ])

        #Add <start> and <end> tokens
        self.df = self.df.with_columns([
            pl.col("en_tokenized").apply(self.add_start_end_tokens).alias("en_final"),
            pl.col("fr_tokenized").apply(self.add_start_end_tokens).alias("fr_final")
        ])

    def save_cleaned_data(self, output_path):
        """
        Saves the cleaned and tokenized data to a CSV file.
        
        Parameters:
        ----------
        output_path : str
            The path to save the cleaned data.
        """
        self.df.write_csv(output_path)
        
    def split_data(self, test_size=0.2, random_state=42):
        """
        Splits the data into training and validation sets.
        
        Parameters:
        ----------
        test_size : float, optional
            The proportion of the data to include in the validation set (default is 0.2).
        random_state : int, optional
            The random seed for reproducibility (default is 42).
        
        Returns:
        ----------
        DataFrame
            The training data.
        DataFrame
            The validation data.
        """
        #Convert to Pandas DataFrame to use train_test_split
        df_selected_pd = self.df.to_pandas()
        
        #Split the data into training and validation sets
        train_df, validation_df = train_test_split(df_selected_pd, test_size=test_size, random_state=random_state)
        
        #Convert back to Polars DataFrame
        train_df_pl = pl.DataFrame(train_df)
        validation_df_pl = pl.DataFrame(validation_df)
        
        return train_df_pl, validation_df_pl

    def save_split_data(self, train_path, validation_path):
        """
        Splits the data and saves the training and validation sets to CSV files.
        
        Parameters:
        ----------
        train_path : str
            The path to save the training data.
        validation_path : str
            The path to save the validation data.
        """
        train_df, validation_df = self.split_data()
        train_df.write_csv(train_path)
        validation_df.write_csv(validation_path)

#Usage example
if __name__ == "__main__":
    #Initialize the DataPreparation class with the path to the data file
    data_prep = DataPreparation("C:/Users/gaye/eng_-french.csv")
    
    #Prepare the data by cleaning, tokenizing, and adding <start> and <end> tokens
    data_prep.prepare_data()
    
    #Save the cleaned data to a new CSV file
    data_prep.save_cleaned_data("C:/Users/gaye/base_cleaned.csv")
    
    #Split the data into training and validation sets and save them to CSV files
    data_prep.save_split_data("C:/Users/gaye/yu.csv", "C:/Users/gaye/yup.csv")


In [2]:
# Import cleaned data
train_data = pd.read_csv("C:/Users/gaye/yu.csv", sep=",", encoding="latin-1", error_bad_lines=False)
validation_data = pd.read_csv("C:/Users/gaye/yup.csv", sep=",", encoding="latin-1", error_bad_lines=False)
train_data.dropna(inplace=True)
validation_data.dropna(inplace=True)

C:\Users\gaye\AppData\Local\Temp\ipykernel_19256\2362623549.py:2: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  train_data = pd.read_csv("C:/Users/gaye/yu.csv", sep=",", encoding="latin-1", error_bad_lines=False)
b'Skipping line 142: expected 8 fields, saw 9\nSkipping line 234: expected 8 fields, saw 9\nSkipping line 525: expected 8 fields, saw 9\nSkipping line 717: expected 8 fields, saw 9\nSkipping line 1318: expected 8 fields, saw 9\nSkipping line 1510: expected 8 fields, saw 9\nSkipping line 1588: expected 8 fields, saw 9\nSkipping line 1613: expected 8 fields, saw 9\nSkipping line 2070: expected 8 fields, saw 9\nSkipping line 2111: expected 8 fields, saw 9\nSkipping line 2262: expected 8 fields, saw 9\nSkipping line 2305: expected 8 fields, saw 10\nSkipping line 2552: expected 8 fields, saw 9\nSkipping line 2638: expected 8 fields, saw 9\nSkipping line 3046: expected 8 fields, saw 9\nSki

# RNN with attention

# Preparing data for training

In [4]:

def preprocess_data(df, source_col, target_col):
    """
    Extracts the source and target texts from the dataframe.
    
    Parameters:
    ----------
    df : DataFrame
        The dataframe containing the data.
    source_col : str
        The name of the column containing the source language texts.
    target_col : str
        The name of the column containing the target language texts.
        
    Returns:
    ----------
    np.array
        The source texts.
    np.array
        The target texts.
    """
    source_texts = df[source_col].values
    target_texts = df[target_col].values
    return source_texts, target_texts


train_source_texts, train_target_texts = preprocess_data(train_data, 'en_final', 'fr_final')
val_source_texts, val_target_texts = preprocess_data(validation_data, 'en_final', 'fr_final')

#Tokenization
source_tokenizer = tf.keras.preprocessing.text.Tokenizer()
source_tokenizer.fit_on_texts(train_source_texts)

target_tokenizer = tf.keras.preprocessing.text.Tokenizer()
target_tokenizer.fit_on_texts(train_target_texts)

#Adding <start> and <end> tokens to the target tokenizer's dictionary
target_tokenizer.word_index['<start>'] = len(target_tokenizer.word_index) + 1
target_tokenizer.word_index['<end>'] = len(target_tokenizer.word_index) + 1

#Verifying that <start> and <end> tokens are present
assert '<start>' in target_tokenizer.word_index
assert '<end>' in target_tokenizer.word_index

train_source_sequences = source_tokenizer.texts_to_sequences(train_source_texts)
train_target_sequences = target_tokenizer.texts_to_sequences(train_target_texts)

val_source_sequences = source_tokenizer.texts_to_sequences(val_source_texts)
val_target_sequences = target_tokenizer.texts_to_sequences(val_target_texts)

max_source_len = max([len(seq) for seq in train_source_sequences])
max_target_len = max([len(seq) for seq in train_target_sequences])

train_source_sequences = tf.keras.preprocessing.sequence.pad_sequences(train_source_sequences, maxlen=max_source_len, padding='post')
train_target_sequences = tf.keras.preprocessing.sequence.pad_sequences(train_target_sequences, maxlen=max_target_len, padding='post')

val_source_sequences = tf.keras.preprocessing.sequence.pad_sequences(val_source_sequences, maxlen=max_source_len, padding='post')
val_target_sequences = tf.keras.preprocessing.sequence.pad_sequences(val_target_sequences, maxlen=max_target_len, padding='post')

In [ ]:

#Model parameters (chosen arbitrarily)
embedding_dim = 256
units = 512
vocab_source_size = len(source_tokenizer.word_index) + 1
vocab_target_size = len(target_tokenizer.word_index) + 1

def create_model():
    """
    Creates and returns a sequence-to-sequence model with attention for translation.
    
    Returns:
    ----------
    Model
        The compiled Keras model.
    """
    #Encoder
    encoder_inputs = Input(shape=(None,))
    encoder_embedding = Embedding(vocab_source_size, embedding_dim)(encoder_inputs)
    encoder_lstm = LSTM(units, return_sequences=True, return_state=True)
    encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

    #Decoder
    decoder_inputs = Input(shape=(None,))
    decoder_embedding = Embedding(vocab_target_size, embedding_dim)(decoder_inputs)
    decoder_lstm = LSTM(units, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=[state_h, state_c])

    #Attention mechanism
    attention = Attention()([decoder_outputs, encoder_outputs])
    concat_attention = Concatenate(axis=-1)([decoder_outputs, attention])

    #Dense layer
    dense = Dense(vocab_target_size, activation='softmax')
    outputs = dense(concat_attention)

    model = Model([encoder_inputs, decoder_inputs], outputs)
    return model

#Create and compile the model
model = create_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

#Prepare the targets for training (shifted sequences)
train_target_sequences_input = train_target_sequences[:, :-1]
train_target_sequences_output = train_target_sequences[:, 1:]

val_target_sequences_input = val_target_sequences[:, :-1]
val_target_sequences_output = val_target_sequences[:, 1:]

#Train the model
history = model.fit(
    [train_source_sequences, train_target_sequences_input],
    train_target_sequences_output,
    validation_data=([val_source_sequences, val_target_sequences_input], val_target_sequences_output),
    batch_size=64,
    epochs=10
)

#Save the model
model.save("C:/Users/gaye/rnn_attention_model.h5")

print("Model training complete and saved.")


In [6]:
model = load_model("C:/Users/gaye/rnn_attention_model.h5")


#Display the summary of the model to get the layer names and architecture
def display_model_summary(model):
    """
    Displays the summary of the Keras model, including layer names and their details.

    Parameters:
    ----------
    model : Model
        The Keras model to summarize.
    """
    model.summary()

#Display the model summary
display_model_summary(model)

#Alternatively, iterate over the model layers to print their names
def print_layer_names(model):
    """
    Iterates over the model layers and prints their names.

    Parameters:
    ----------
    model : Model
        The Keras model whose layer names are to be printed.
    """
    for layer in model.layers:
        print(layer.name)



Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_5       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 256) │  2,096,384 │ input_layer_4[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 256) │  4,185,344 │ input_layer_5[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, None,     │  1,574,912 │ embedding_2[0][0] │
│                     │ 512), (None,      │            │                   │
│                     │ 512), (None,      │            │                   │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, None,     │  1,574,912 │ embedding_3[0][0… │
│                     │ 512), (None,      │            │ lstm_2[0][1],     │
│                     │ 512), (None,      │            │ lstm_2[0][2]      │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_1         │ (None, None, 512) │          0 │ lstm_3[0][0],     │
│ (Attention)         │                   │            │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, None,      │          0 │ lstm_3[0][0],     │
│ (Concatenate)       │ 1024)             │            │ attention_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None,      │ 16,757,725 │ concatenate_1[0]… │
│                     │ 16349)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 26,189,279 (99.90 MB)

 Trainable params: 26,189,277 (99.90 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

# Translation process

In [19]:
#Load the saved model
model = load_model("C:/Users/gaye/rnn_attention_model.h5")

#Example test sentence
example_test = "How are you doing"

def preprocess_test_example(example, tokenizer, max_len):
    """
    Preprocesses a test example by tokenizing and padding the input sentence.
    
    Parameters:
    ----------
    example : str
        The input sentence to be translated.
    tokenizer : Tokenizer
        The tokenizer used to convert the sentence to a sequence of tokens.
    max_len : int
        The maximum length for padding the sequence.
        
    Returns:
    ----------
    np.array
        The tokenized and padded sequence.
    """
    example_sequence = tokenizer.texts_to_sequences([example])
    example_padded = tf.keras.preprocessing.sequence.pad_sequences(example_sequence, maxlen=max_len, padding='post')
    return example_padded

#Tokenization
source_tokenizer = tf.keras.preprocessing.text.Tokenizer()
target_tokenizer = tf.keras.preprocessing.text.Tokenizer()
source_tokenizer.fit_on_texts(train_data['en_final'])
target_tokenizer.fit_on_texts(train_data['fr_final'])

#Manually add <start> and <end> tokens to the target tokenizer's dictionary
target_tokenizer.word_index['<start>'] = len(target_tokenizer.word_index) + 1
target_tokenizer.word_index['<end>'] = len(target_tokenizer.word_index) + 1

#Add these tokens to the index word mapping as well
target_tokenizer.index_word[target_tokenizer.word_index['<start>']] = '<start>'
target_tokenizer.index_word[target_tokenizer.word_index['<end>']] = '<end>'

#Calculate the maximum length of source and target sequences
max_source_len = max(train_data['en_final'].apply(lambda x: len(x.split())))
max_target_len = max(train_data['fr_final'].apply(lambda x: len(x.split())))

#Preprocess the example test sentence
example_padded = preprocess_test_example(example_test, source_tokenizer, max_source_len)

#Reconstruct the encoder model using the exact layer names
encoder_inputs = Input(shape=(max_source_len,), name="input_layer_5")
encoder_embedding_layer = model.get_layer('embedding_2')
encoder_lstm_layer = model.get_layer('lstm_2')
encoder_embedding = encoder_embedding_layer(encoder_inputs)
encoder_outputs, state_h, state_c = encoder_lstm_layer(encoder_embedding)

encoder_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

#Reconstruct the decoder model
decoder_inputs = Input(shape=(None,), name="input_layer_6")
decoder_state_input_h = Input(shape=(512,), name="input_3")
decoder_state_input_c = Input(shape=(512,), name="input_4")
decoder_hidden_state_input = Input(shape=(max_source_len, 512))

decoder_embedding_layer = model.get_layer('embedding_3')
decoder_lstm_layer = model.get_layer('lstm_3')
attention_layer = model.get_layer('attention_1')
concat_layer = model.get_layer('concatenate_1')
dense_layer = model.get_layer('dense_1')

decoder_embedding = decoder_embedding_layer(decoder_inputs)
decoder_outputs, state_h, state_c = decoder_lstm_layer(decoder_embedding, initial_state=[decoder_state_input_h, decoder_state_input_c])
attention = attention_layer([decoder_outputs, decoder_hidden_state_input])
decoder_concat_input = concat_layer([decoder_outputs, attention])
decoder_outputs = dense_layer(decoder_concat_input)

decoder_model = Model(
    [decoder_inputs] + [decoder_hidden_state_input, decoder_state_input_h, decoder_state_input_c],
    [decoder_outputs] + [state_h, state_c]
)

def translate_sequence(input_seq):
    """
    Generates a translation for the input sequence using the trained model.
    
    Parameters:
    ----------
    input_seq : np.array
        The tokenized and padded input sequence.
        
    Returns:
    ----------
    str
        The translated sentence.
    """
    #Encode the input sequence to get the context vectors
    enc_out, enc_h, enc_c = encoder_model.predict(input_seq)

    #Initialize the target sequence with the <start> token
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index['<start>']

    stop_condition = False
    decoded_sentence = []

    while not stop_condition:
        #Predict the next token and the decoder states
        dec_out, dec_h, dec_c = decoder_model.predict(
            [target_seq] + [enc_out, enc_h, enc_c]
        )

        #Get the most likely next token
        sampled_token_index = np.argmax(dec_out[0, -1, :])
        sampled_word = target_tokenizer.index_word.get(sampled_token_index, '')

        #Check for the end of the sentence or the maximum length
        if sampled_word == '<end>' or len(decoded_sentence) >= max_target_len:
            stop_condition = True
        else:
            decoded_sentence.append(sampled_word)

        #Update the target sequence
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        #Update the encoder states
        enc_h, enc_c = dec_h, dec_c

    return ' '.join(decoded_sentence)

In [17]:
#Translate the example test sentence
translated_text = translate_sequence(example_padded)
print("Translated text:", translated_text)

# Baseline comparison with RNN constructed with LSTMs units : 

# Construction of RNN with LSTM units : 

In [ ]:

#Model parameters (chosen arbitrarily)
embedding_dim = 256
units = 512
vocab_source_size = len(source_tokenizer.word_index) + 1
vocab_target_size = len(target_tokenizer.word_index) + 1

def create_model():
    """
    Creates and returns a sequence-to-sequence model for translation without attention.
    
    Returns:
    ----------
    Model
        The compiled Keras model.
    """
    #Encoder
    encoder_inputs = Input(shape=(None,))
    encoder_embedding = Embedding(vocab_source_size, embedding_dim)(encoder_inputs)
    encoder_lstm = LSTM(units, return_state=True)
    _, state_h, state_c = encoder_lstm(encoder_embedding)

    #Decoder
    decoder_inputs = Input(shape=(None,))
    decoder_embedding = Embedding(vocab_target_size, embedding_dim)(decoder_inputs)
    decoder_lstm = LSTM(units, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=[state_h, state_c])

    #Dense layer
    dense = Dense(vocab_target_size, activation='softmax')
    outputs = dense(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], outputs)
    return model

#Create and compile the model
model = create_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

#Prepare the targets for training (shifted sequences)
train_target_sequences_input = train_target_sequences[:, :-1]
train_target_sequences_output = train_target_sequences[:, 1:]

val_target_sequences_input = val_target_sequences[:, :-1]
val_target_sequences_output = val_target_sequences[:, 1:]

#Train the model
history = model.fit(
    [train_source_sequences, train_target_sequences_input],
    train_target_sequences_output,
    validation_data=([val_source_sequences, val_target_sequences_input], val_target_sequences_output),
    batch_size=64,
    epochs=10
)

#Save the model
model.save("C:/Users/gaye/rnn_lstm_model.h5")

print("Model training complete and saved.")

In [7]:
model_without_attention = tf.keras.models.load_model("C:/Users/gaye/rnn_lstm_model.h5")
model_without_attention.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 256) │  2,096,384 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 256) │  4,185,344 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 512),     │  1,574,912 │ embedding_2[0][0] │
│                     │ (None, 512),      │            │                   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, None,     │  1,574,912 │ embedding_3[0][0… │
│                     │ 512), (None,      │            │ lstm_2[0][1],     │
│                     │ 512), (None,      │            │ lstm_2[0][2]      │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None,      │  8,387,037 │ lstm_3[0][0]      │
│                     │ 16349)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 17,818,591 (67.97 MB)

 Trainable params: 17,818,589 (67.97 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

# Model comparison

In [8]:
#Load model with attention
model_with_attention = tf.keras.models.load_model("C:/Users/gaye/rnn_attention_model.h5")

#Load model without attention
model_without_attention = tf.keras.models.load_model("C:/Users/gaye/rnn_lstm_model.h5")

In [10]:
#Prepare the test data

#The preprocess_data function extracts the source and target texts from the validation_data DataFrame.
test_source_texts, test_target_texts = preprocess_data(validation_data, 'en_final', 'fr_final')

#Convert the source texts into sequences of tokens using the tokenizer trained on the training data.
test_source_sequences = source_tokenizer.texts_to_sequences(test_source_texts)
#Convert the target texts into sequences of tokens using the tokenizer trained on the training data.
test_target_sequences = target_tokenizer.texts_to_sequences(test_target_texts)

#Pad the source sequences to ensure they all have the same length (max_source_len), using 'post' padding.
test_source_sequences = tf.keras.preprocessing.sequence.pad_sequences(test_source_sequences, maxlen=max_source_len, padding='post')
#Pad the target sequences to ensure they all have the same length (max_target_len), using 'post' padding.
test_target_sequences = tf.keras.preprocessing.sequence.pad_sequences(test_target_sequences, maxlen=max_target_len, padding='post')


# Loss & Accuracy

In [13]:
#Model with attention evaluation
loss_with_attention, accuracy_with_attention = model_with_attention.evaluate([test_source_sequences, test_target_sequences[:, :-1]], test_target_sequences[:, 1:])
print(f"Modele with attention - Loss: {loss_with_attention}, Precision: {accuracy_with_attention}")

#Model without attention evaluation
loss_without_attention, accuracy_without_attention = model_without_attention.evaluate([test_source_sequences, test_target_sequences[:, :-1]], test_target_sequences[:, 1:])
print(f"Modele without attention - Loss: {loss_without_attention}, Precision: {accuracy_without_attention}")


600/600 ━━━━━━━━━━━━━━━━━━━━ 74s 123ms/step - accuracy: 0.8882 - loss: 0.6167
Modele with attention - Loss: 0.6240121722221375, Precision: 0.8875821232795715
600/600 ━━━━━━━━━━━━━━━━━━━━ 64s 106ms/step - accuracy: 0.8791 - loss: 0.6434
Modele without attention - Loss: 0.6499518752098083, Precision: 0.8785788416862488


# Translation comparison 

In [14]:
#Reconstruct the encoder model without attention
encoder_inputs_wo_attention = Input(shape=(max_source_len,), name="input_layer_5")
encoder_embedding_layer_wo_attention = model_without_attention.get_layer('embedding_2')
encoder_lstm_layer_wo_attention = model_without_attention.get_layer('lstm_2')
encoder_embedding_wo_attention = encoder_embedding_layer_wo_attention(encoder_inputs_wo_attention)
_, state_h_wo_attention, state_c_wo_attention = encoder_lstm_layer_wo_attention(encoder_embedding_wo_attention)

encoder_model_without_attention = Model(encoder_inputs_wo_attention, [state_h_wo_attention, state_c_wo_attention])

#Reconstruct the decoder model without attention
decoder_inputs_wo_attention = Input(shape=(None,), name="input_layer_6")
decoder_state_input_h_wo_attention = Input(shape=(512,), name="input_3")
decoder_state_input_c_wo_attention = Input(shape=(512,), name="input_4")

decoder_embedding_layer_wo_attention = model_without_attention.get_layer('embedding_3')
decoder_lstm_layer_wo_attention = model_without_attention.get_layer('lstm_3')
dense_layer_wo_attention = model_without_attention.get_layer('dense_1')

decoder_embedding_wo_attention = decoder_embedding_layer_wo_attention(decoder_inputs_wo_attention)
decoder_outputs_wo_attention, state_h_wo_attention, state_c_wo_attention = decoder_lstm_layer_wo_attention(
    decoder_embedding_wo_attention, initial_state=[decoder_state_input_h_wo_attention, decoder_state_input_c_wo_attention])
decoder_outputs_wo_attention = dense_layer_wo_attention(decoder_outputs_wo_attention)

decoder_model_without_attention = Model(
    [decoder_inputs_wo_attention] + [decoder_state_input_h_wo_attention, decoder_state_input_c_wo_attention],
    [decoder_outputs_wo_attention] + [state_h_wo_attention, state_c_wo_attention]
)


In [15]:
def decode_sequence_without_attention(input_seq):
    """
    Decodes a given input sequence into a translated sentence using the trained model without attention.

    Parameters:
    ----------
    input_seq : np.array
        The tokenized and padded input sequence to be translated.

    Returns:
    ----------
    str
        The translated sentence.
    """
    #Encode the input as state vectors using the encoder model without attention.
    states_value = encoder_model_without_attention.predict(input_seq)

    #Generate an empty target sequence of length 1.
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index['<start>']

    stop_condition = False
    decoded_sentence = []

    while not stop_condition:
        #Predict the next token and the decoder states using the decoder model without attention.
        output_tokens, h, c = decoder_model_without_attention.predict([target_seq] + states_value)

        #Get the most likely next token.
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = target_tokenizer.index_word.get(sampled_token_index, '')

        #Check for the end of the sentence or the maximum length.
        if sampled_word == '<end>' or len(decoded_sentence) >= max_target_len:
            stop_condition = True
        else:
            decoded_sentence.append(sampled_word)

        #Update the target sequence.
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        #Update the states.
        states_value = [h, c]

    #Return the decoded sentence as a single string.
    return ' '.join(decoded_sentence)


In [74]:
example_test = "You can not hear me"

In [75]:
#Preprocess the example test sentence
example_padded = preprocess_test_example(example_test, source_tokenizer, max_source_len)

#Translate the example test sentence with attention
translated_text_with_attention = translate_sequence(example_padded)


#Translate the example test sentence without attention
translated_text_without_attention = decode_sequence_without_attention(example_padded)
print("Sentence to translate : ", example_test)
print("Translated text with attention:", translated_text_with_attention)
print("Translated text without attention:", translated_text_without_attention)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━